# Lab 10: Conditional GAN on MNIST

**Goal:** Build a GAN that generates a **specific digit** on demand.

Key ideas:
- Feed labels to **both** Generator and Discriminator
- Generator: concatenate label embedding with noise
- Discriminator: add label as extra image channels
- After training: "Generate a 7" actually produces a 7!

**Expected compute** (approximate): 30 + 10 epochs of a small DCGAN ≈ 10 min on Apple-silicon (`mps`), ≈ 5–8 min on a Colab T4, ≈ 15 min on CPU with the automatic 10k-image subset. The trained generator is saved to `cgan_generator.pt` so Lab 11 can skip retraining.

**How to use this lab:** answer each **Predict** prompt before running the cell.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else "cpu")
print(f"Using: {device}")

## 1. Data

This time we **keep the labels** — we need them for conditioning!

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

mnist = datasets.MNIST("data", train=True, download=True, transform=transform)

# CPU-only machines train on a 10k-image subset (None = all 60k images, used on GPU / mps).
TRAIN_SUBSET = 10_000 if device.type == "cpu" else None
if TRAIN_SUBSET:
    mnist = torch.utils.data.Subset(mnist, range(TRAIN_SUBSET))
loader = DataLoader(mnist, batch_size=64, shuffle=True)

## 2. Conditional Generator

Takes **noise + label** and produces an image.

```
noise (64-dim) ──┐
                  ├── concat (114-dim) ──> ConvTranspose layers ──> 1x28x28
label embed (50) ─┘
```

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=64, num_classes=10, embed_dim=50):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, embed_dim)

        self.model = nn.Sequential(
            nn.Linear(z_dim + embed_dim, 256 * 7 * 7),
            nn.Unflatten(1, (256, 7, 7)),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),   # -> 128 x 14 x 14
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 1, 4, 2, 1),     # -> 1 x 28 x 28
            nn.Tanh(),
        )

    def forward(self, z, labels):
        label_vec = self.label_embed(labels)          # (batch, 50)
        x = torch.cat([z, label_vec], dim=1)          # (batch, 114)
        return self.model(x)

print("Generator: takes noise + label, produces image")

## 3. Conditional Discriminator

Takes **image + label** and decides real/fake.

```
image (1x28x28) ──────┐
                       ├── concat (11x28x28) ──> Conv layers ──> real/fake
label channels (10x28x28) ┘
```

The label is turned into 10 image-sized channels (one-hot expanded).

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.num_classes = num_classes

        self.model = nn.Sequential(
            # Input: (1 + 10) x 28 x 28 = 11 channels
            nn.Conv2d(1 + num_classes, 64, 4, 2, 1),   # -> 64 x 14 x 14
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),                # -> 128 x 7 x 7
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, images, labels):
        label_maps = self._make_label_maps(labels, images.size(2), images.size(3))
        x = torch.cat([images, label_maps], dim=1)     # (batch, 11, 28, 28)
        return self.model(x)

    def _make_label_maps(self, labels, h, w):
        """Turn integer labels into one-hot image channels."""
        bs = labels.size(0)
        one_hot = torch.zeros(bs, self.num_classes, device=labels.device)
        one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
        return one_hot.view(bs, self.num_classes, 1, 1).expand(-1, -1, h, w)

print("Discriminator: takes image + label, checks if they match")

### Quick test: label maps

Let's see what `_make_label_maps` produces.

In [ ]:
# Test: visualize label maps for digit 7
disc_test = Discriminator()
test_labels = torch.tensor([7])
label_maps = disc_test._make_label_maps(test_labels, 28, 28)
print(f"Label maps shape: {label_maps.shape}")  # (1, 10, 28, 28)

fig, axes = plt.subplots(1, 10, figsize=(15, 1.5))
for i in range(10):
    axes[i].imshow(label_maps[0, i].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(f"Ch {i}", fontsize=9)
    axes[i].axis("off")
plt.suptitle("Label channels for digit 7 (channel 7 is all-white, rest all-black)", fontsize=11)
plt.tight_layout()
plt.show()
del disc_test

## 4. Setup & Training

In [ ]:
z_dim       = 64
num_classes = 10
lr          = 2e-4
epochs      = 30

gen  = Generator(z_dim, num_classes).to(device)
disc = Discriminator(num_classes).to(device)

opt_G = torch.optim.Adam(gen.parameters(),  lr=lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))
criterion = nn.BCELoss()

print(f"Generator params:     {sum(p.numel() for p in gen.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in disc.parameters()):,}")

In [ ]:
history_D = []
history_G = []

for epoch in range(epochs):
    ep_D, ep_G = [], []

    for real_images, real_labels in loader:
        real_images = real_images.to(device)
        real_labels = real_labels.to(device)
        bs = real_images.size(0)

        real_target = torch.ones(bs, 1, device=device)
        fake_target = torch.zeros(bs, 1, device=device)

        # ── Train Discriminator ──
        noise = torch.randn(bs, z_dim, device=device)
        fake_images = gen(noise, real_labels).detach()

        pred_real = disc(real_images, real_labels)    # real image + correct label
        pred_fake = disc(fake_images, real_labels)    # fake image + same label

        loss_D = criterion(pred_real, real_target) + criterion(pred_fake, fake_target)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # ── Train Generator ──
        noise = torch.randn(bs, z_dim, device=device)
        fake_images = gen(noise, real_labels)
        pred_fake = disc(fake_images, real_labels)

        loss_G = criterion(pred_fake, real_target)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        ep_D.append(loss_D.item())
        ep_G.append(loss_G.item())

    avg_D = sum(ep_D) / len(ep_D)
    avg_G = sum(ep_G) / len(ep_G)
    history_D.append(avg_D)
    history_G.append(avg_G)

    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:3d} | D: {avg_D:.4f} | G: {avg_G:.4f}")

# Save the generator so Lab 11 (controllable generation) can load it instead of retraining
torch.save(gen.state_dict(), "cgan_generator.pt")
print("Saved generator to cgan_generator.pt")

## 5. Loss Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_D, label="Discriminator")
ax1.set_title("Discriminator Loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_G, label="Generator", color="orange")
ax2.set_title("Generator Loss")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. The Payoff: Generate Any Digit!

Each row = one digit (0-9).  
Each column = different noise (different style).  
**Same noise, different label = different digit. Same label, different noise = different style.**

**Predict:** which digits will look worst after 30 epochs?

In [ ]:
gen.eval()
fig, axes = plt.subplots(10, 8, figsize=(10, 13))

with torch.no_grad():
    for digit in range(10):
        noise = torch.randn(8, z_dim, device=device)
        labels = torch.full((8,), digit, dtype=torch.long, device=device)
        fakes = gen(noise, labels).cpu()

        for j in range(8):
            img = (fakes[j].squeeze() + 1) / 2
            axes[digit][j].imshow(img, cmap="gray")
            axes[digit][j].axis("off")
        axes[digit][0].text(-0.08, 0.5, str(digit), transform=axes[digit][0].transAxes, ha='right', va='center', fontsize=14, fontweight='bold')

plt.suptitle("Conditional GAN: Each Row = One Digit", fontsize=14)
plt.tight_layout()
plt.show()
gen.train();

## 7. Experiment: Same Noise, Different Labels

What happens when we keep the noise fixed but change the label?  
**Predict:** will the stroke thickness / slant stay the same across the row?

*Expected:* the **style stays similar** but the **digit changes**.

In [ ]:
gen.eval()
fig, axes = plt.subplots(3, 10, figsize=(12, 4))

with torch.no_grad():
    for row in range(3):
        # SAME noise for all 10 digits
        fixed = torch.randn(1, z_dim, device=device).expand(10, -1)
        labels = torch.arange(10, device=device)
        fakes = gen(fixed, labels).cpu()

        for j in range(10):
            img = (fakes[j].squeeze() + 1) / 2
            axes[row][j].imshow(img, cmap="gray")
            axes[row][j].axis("off")
            if row == 0:
                axes[row][j].set_title(str(j), fontsize=12)

plt.suptitle("Same Noise, Different Labels (style is preserved across digits)", fontsize=13)
plt.tight_layout()
plt.show()
gen.train();

## 8. Experiment: What if D Doesn't Get the Label?

Let's prove why D needs the label. We'll train a "broken" cGAN where only G gets the label.

**Predict:** G still receives the label. Will its outputs follow the label anyway?

In [ ]:
class BrokenDiscriminator(nn.Module):
    """D without label — can't check if image matches the request."""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, images, labels=None):   # labels ignored!
        return self.model(images)

# Train for just 10 epochs to see the problem
gen_broken  = Generator(z_dim, num_classes).to(device)
disc_broken = BrokenDiscriminator().to(device)
opt_Gb = torch.optim.Adam(gen_broken.parameters(),  lr=2e-4, betas=(0.5, 0.999))
opt_Db = torch.optim.Adam(disc_broken.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Training 'broken' cGAN (D ignores labels) for 10 epochs...")
for epoch in range(10):
    for real_images, real_labels in loader:
        real_images = real_images.to(device)
        real_labels = real_labels.to(device)
        bs = real_images.size(0)

        real_t = torch.ones(bs, 1, device=device)
        fake_t = torch.zeros(bs, 1, device=device)

        noise = torch.randn(bs, z_dim, device=device)
        fake = gen_broken(noise, real_labels).detach()
        loss_D = criterion(disc_broken(real_images), real_t) + criterion(disc_broken(fake), fake_t)
        opt_Db.zero_grad(); loss_D.backward(); opt_Db.step()

        noise = torch.randn(bs, z_dim, device=device)
        fake = gen_broken(noise, real_labels)
        loss_G = criterion(disc_broken(fake), real_t)
        opt_Gb.zero_grad(); loss_G.backward(); opt_Gb.step()

    print(f"  Epoch {epoch}")

print("Done. Let's see if it respects labels...")

In [ ]:
gen_broken.eval()
fig, axes = plt.subplots(2, 10, figsize=(12, 3))

with torch.no_grad():
    for row in range(2):
        fixed = torch.randn(1, z_dim, device=device).expand(10, -1)
        labels = torch.arange(10, device=device)
        fakes = gen_broken(fixed, labels).cpu()

        for j in range(10):
            img = (fakes[j].squeeze() + 1) / 2
            axes[row][j].imshow(img, cmap="gray")
            axes[row][j].axis("off")
            if row == 0:
                axes[row][j].set_title(f"label={j}", fontsize=9)

plt.suptitle("Broken cGAN: D ignores labels → G ignores them too!", fontsize=13)
plt.tight_layout()
plt.show()

print("Notice: the digits DON'T match the labels!")
print("Without label info, D can't enforce label-matching.")
print("G learns that labels don't matter and ignores them.")

## Key Takeaways

1. **Feed labels to BOTH G and D** — D must check label-image match
2. **G: embedding + concat** — label becomes a learned vector, merged with noise
3. **D: one-hot channels** — label becomes extra image channels
4. **Noise = style, Label = what** — independent controls
5. **Without label in D** — G ignores labels (proven in experiment!)

**Next: Lab 11 (Controllable Generation)** — manipulate the latent space for fine-grained control!